For each file, find its person id and split it into 5 second chunks

In [2]:
import os
import pandas as pd
import numpy as np
import time 

kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_3/updated_data_kalman_filtered_3_test_reserve'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'

out_path = '../data/kaggle-drdataboston/attempt_4'

X = []
y = []

window_ms = 5000
interp_pts = 500

matrix = pd.read_csv(matrix_path)

start_time = time.perf_counter()

for idx, file in enumerate(os.listdir(kalman_filter_data_path)):
    if idx % 5 == 0:
        c_time = time.perf_counter()
        print(f"Processed {idx} files in {c_time - start_time:.6f} seconds")

    if not file.endswith('.csv'):
        continue

    file_path = os.path.join(kalman_filter_data_path, file)
    df = pd.read_csv(file_path)


    clean_name = file
    if(clean_name.endswith('.csv.csv')):
        clean_name = clean_name[:-4]
    
    person_id = None
    for idx, row in matrix.iterrows():
        if clean_name.strip().lower() in [str(row.iloc[i]).strip().lower() for i in range(1, 5)]:
            person_id = row.iloc[0]
            break

    if person_id is None:
        continue

    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms

    chunks = []
    labels = []

    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue

        time_arr = group['timestamp'].values
        time_arr = time_arr - time_arr[0]

        new_time = np.linspace(0, time_arr[-1], interp_pts)

        x_interp = np.interp(new_time, time_arr, group['acc_x_kf'].values)
        y_interp = np.interp(new_time, time_arr, group['acc_y_kf'].values)
        z_interp = np.interp(new_time, time_arr, group['acc_z_kf'].values)

        # Combine into shape (500, 3)
        interp_window = np.stack((x_interp, y_interp, z_interp), axis=-1)

        chunks.append(interp_window)
        labels.append(person_id)

    X.extend(chunks)
    y.extend(labels)


    print(np.array(X).shape)

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)
    

np.save(f"{out_path}/timewise_5s_500p_reserve_X.npy", X)
np.save(f"{out_path}/timewise_5s_500p_reserve_y.npy", y)



Processed 0 files in 0.000831 seconds
(81, 500, 3)
(186, 500, 3)
(281, 500, 3)
(376, 500, 3)
(469, 500, 3)
Processed 5 files in 0.353825 seconds
(560, 500, 3)
(657, 500, 3)
(734, 500, 3)
(834, 500, 3)
(919, 500, 3)
Processed 10 files in 0.680403 seconds
(1016, 500, 3)
(1111, 500, 3)
(1201, 500, 3)
(1304, 500, 3)
(1398, 500, 3)
Processed 15 files in 1.016496 seconds
(1485, 500, 3)
(1588, 500, 3)
(1697, 500, 3)
(1786, 500, 3)
(1881, 500, 3)
Processed 20 files in 1.358550 seconds
(1993, 500, 3)
(2088, 500, 3)
(2181, 500, 3)
(2294, 500, 3)
(2389, 500, 3)
Processed 25 files in 1.760702 seconds
(2488, 500, 3)
(2581, 500, 3)
(2679, 500, 3)
(2756, 500, 3)
(2851, 500, 3)
(2851, 500, 3) (2851,)


In [4]:
from collections import Counter

# Count occurrences of each label
label_counts = Counter(y)

# Find label with the fewest entries
min_label, min_count = min(label_counts.items(), key=lambda x: x[1])

max_label, max_count = max(label_counts.items(), key=lambda x: x[1])

print(f"\nLabel with fewest samples: {min_label} ({min_count} samples)")
print(f"Label with most samples: {max_label} ({max_count} samples)")



Label with fewest samples: 47 (87 samples)
Label with most samples: 51 (457 samples)


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


2025-05-02 10:45:52.829301: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 10:45:52.839802: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 10:45:52.922771: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 10:45:52.987452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746171953.044360   17319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746171953.06

In [6]:
data_path = '../data/kaggle-drdataboston/attempt_3'
fname = 'timewise_5s_500p'

X = np.load(f'{data_path}/{fname}_X.npy')  # shape: (N, T)
y = np.load(f'{data_path}/{fname}_y.npy')  # shape: (N,)

print(X.shape, y.shape)


(26088, 500, 3) (26088,)


In [7]:
import joblib
from sklearn.preprocessing import LabelEncoder
# Fit on the full set of labels (before train/test split)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

joblib.dump(label_encoder, f'{data_path}/label_encoder.pkl')

# Then split the encoded labels
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [8]:
# X shape: (N, T, 3)
X_train_scaled = np.zeros_like(X_train)
X_val_scaled = np.zeros_like(X_val)
X_test_scaled = np.zeros_like(X_test)

scalers = []

for i in range(X_train.shape[2]):  # loop over channels: 0=x, 1=y, 2=z
    scaler = StandardScaler()
    scaler.fit(X_train[:, :, i])  # fit on (N, T) for this channel
    
    X_train_scaled[:, :, i] = scaler.transform(X_train[:, :, i])
    X_val_scaled[:, :, i]   = scaler.transform(X_val[:, :, i])
    X_test_scaled[:, :, i]  = scaler.transform(X_test[:, :, i])
    
    scalers.append(scaler)

# Optionally save all 3 scalers
joblib.dump(scalers, f'{data_path}/scalers.pkl')


['../data/kaggle-drdataboston/attempt_3/scalers.pkl']

In [9]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_encoded))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [11]:
from sklearn.utils import class_weight
import numpy as np

# Compute weights for each class
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dict for Keras
class_weights_dict = dict(enumerate(class_weights_array))


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input

model3 = Sequential([
    Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


2025-05-02 10:48:35.563794: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


model3.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=30,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop],
    class_weight=class_weights_dict
)


Epoch 1/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.0465 - loss: 4.2876 - val_accuracy: 0.2111 - val_loss: 3.3815
Epoch 2/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.2214 - loss: 2.9683 - val_accuracy: 0.4373 - val_loss: 2.2645
Epoch 3/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.4111 - loss: 2.0361 - val_accuracy: 0.5921 - val_loss: 1.6989
Epoch 4/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.5251 - loss: 1.5502 - val_accuracy: 0.6754 - val_loss: 1.2718
Epoch 5/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.6181 - loss: 1.1865 - val_accuracy: 0.7069 - val_loss: 1.1107
Epoch 6/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.6701 - loss: 1.0074 - val_accuracy: 0.7511 - val_loss: 0.9584
Epoch 7/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.7205 - loss: 0.8421 - val_accuracy: 0.7782 - val_loss: 0.8529
Epoch 8/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 32ms/step - accuracy: 0.7526 - loss: 0.7403 - 

In [17]:
from tensorflow.keras.models import load_model

model_lstm = load_model(f'{data_path}/CNN_LSTM/best_cnn_lstm.keras')

In [18]:
test_loss, test_acc = model_lstm.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9607 - loss: 0.1744
Test accuracy: 0.96


In [16]:
test_loss, test_acc = model3.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

model3.save('../data/kaggle-drdataboston/attempt_3/model3-cnn.keras')


123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8684 - loss: 0.5820
Test accuracy: 0.87
